In [1]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("dmis-lab/biobert-base-cased-v1.2")
print("Tokenizer loaded successfully!")
print("Vocabulary size:", tokenizer.vocab_size)

/Users/angelinagupta/biomedical-nlp-project/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Tokenizer loaded successfully!
Vocabulary size: 28996


In [2]:
# Try a simple biomedical sentence
sentence = "Metformin causes hepatotoxicity in diabetic patients"

tokens = tokenizer.tokenize(sentence)
print("Original sentence:", sentence)
print("Subword tokens:", tokens)
print("Total tokens:", len(tokens))

Original sentence: Metformin causes hepatotoxicity in diabetic patients
Subword tokens: ['met', '##form', '##in', 'causes', 'he', '##pa', '##to', '##to', '##xi', '##city', 'in', 'di', '##abe', '##tic', 'patients']
Total tokens: 15


In [3]:
words = ["hepatotoxicity", "adenomatous", "pharmacokinetics", "aspirin", "diabetes"]

for word in words:
    tokens = tokenizer.tokenize(word)
    print(f"{word:25} → {tokens}")

hepatotoxicity            → ['he', '##pa', '##to', '##to', '##xi', '##city']
adenomatous               → ['ad', '##eno', '##mat', '##ous']
pharmacokinetics          → ['p', '##har', '##ma', '##co', '##kin', '##etics']
aspirin                   → ['as', '##pi', '##rin']
diabetes                  → ['diabetes']


In [4]:
encoded = tokenizer(sentence, return_tensors="pt")

print("Input IDs:", encoded["input_ids"])
print("Tokens with special tokens:", tokenizer.convert_ids_to_tokens(encoded["input_ids"][0]))

Input IDs: tensor([[  101,  1899, 13199,  1394,  4680,  1119,  4163,  2430,  2430,  8745,
          9041,  1107,  4267, 22377,  2941,  4420,   102]])
Tokens with special tokens: ['[CLS]', 'met', '##form', '##in', 'causes', 'he', '##pa', '##to', '##to', '##xi', '##city', 'in', 'di', '##abe', '##tic', 'patients', '[SEP]']


In [5]:
from datasets import load_dataset

PARQUET_BASE = "hf://datasets/ncbi_disease@refs/convert/parquet/ncbi_disease"
dataset = load_dataset("parquet", data_files={
    "train": f"{PARQUET_BASE}/train/0000.parquet",
    "validation": f"{PARQUET_BASE}/validation/0000.parquet",
    "test": f"{PARQUET_BASE}/test/0000.parquet",
})

label_names = ["O", "B-Disease", "I-Disease"]

# Take first sample
sample = dataset["train"][0]
words = sample["tokens"]
labels = sample["ner_tags"]

print("=== Original Word-Level Labels ===")
for word, label in zip(words, labels):
    print(f"{word:20} → {label_names[label]}")

print("\n=== After BioBERT Tokenization ===")
tokens = tokenizer.tokenize(" ".join(words))
for token in tokens:
    print(f"{token:20} → ?")

0000.parquet:   0%|          | 0.00/425k [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/74.7k [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/77.0k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

=== Original Word-Level Labels ===
Identification       → O
of                   → O
APC2                 → O
,                    → O
a                    → O
homologue            → O
of                   → O
the                  → O
adenomatous          → B-Disease
polyposis            → I-Disease
coli                 → I-Disease
tumour               → I-Disease
suppressor           → O
.                    → O

=== After BioBERT Tokenization ===
identification       → ?
of                   → ?
a                    → ?
##p                  → ?
##c                  → ?
##2                  → ?
,                    → ?
a                    → ?
ho                   → ?
##mo                 → ?
##logue              → ?
of                   → ?
the                  → ?
ad                   → ?
##eno                → ?
##mat                → ?
##ous                → ?
p                    → ?
##oly                → ?
##po                 → ?
##sis                → ?
co                   →

In [7]:
def align_labels_with_tokens(words, labels, tokenizer, label_names):
    tokenized = tokenizer(
        words,
        is_split_into_words=True,  # tells tokenizer words are pre-split
        return_offsets_mapping=True,
        padding=False,
        truncation=True,
        max_length=512
    )
    
    aligned_labels = []
    previous_word_id = None
    
    for word_id in tokenized.word_ids():
        if word_id is None:
            # CLS and SEP tokens → ignore
            aligned_labels.append(-100)
        elif word_id != previous_word_id:
            # First subword of a new word → assign real label
            aligned_labels.append(labels[word_id])
        else:
            # Remaining subwords of same word → ignore
            aligned_labels.append(-100)
        previous_word_id = word_id
    
    return tokenized, aligned_labels

# Test on first sample
sample = dataset["train"][0]
words = sample["tokens"]
labels = sample["ner_tags"]

tokenized, aligned_labels = align_labels_with_tokens(words, labels, tokenizer, label_names)

print("Token → Label mapping after alignment:\n")
tokens = tokenizer.convert_ids_to_tokens(tokenized["input_ids"])
for token, label_id in zip(tokens, aligned_labels):
    label_str = label_names[label_id] if label_id != -100 else "IGNORED"
    print(f"{token:20} → {label_str}")

Token → Label mapping after alignment:

[CLS]                → IGNORED
identification       → O
of                   → O
a                    → O
##p                  → IGNORED
##c                  → IGNORED
##2                  → IGNORED
,                    → O
a                    → O
ho                   → O
##mo                 → IGNORED
##logue              → IGNORED
of                   → O
the                  → O
ad                   → B-Disease
##eno                → IGNORED
##mat                → IGNORED
##ous                → IGNORED
p                    → I-Disease
##oly                → IGNORED
##po                 → IGNORED
##sis                → IGNORED
co                   → I-Disease
##li                 → IGNORED
t                    → I-Disease
##umour              → IGNORED
suppress             → O
##or                 → IGNORED
.                    → O
[SEP]                → IGNORED


Here's the key insight — BERT never looks at subword tokens in isolation. It looks at all tokens together through attention. So when BERT sees ad, it simultaneously sees ##eno, ##mat, ##ous around it and understands that together they form one medical word. The attention mechanism connects them.

A simple analogy:
Imagine you're grading a team project. The team has 4 members but you give one grade to the team lead that represents the whole team's work. You don't grade each member separately — that would be redundant. The -100 is exactly this — you're saying "this subword is part of the same team, don't grade it separately."

In [8]:
def tokenize_and_align_dataset(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        is_split_into_words=True,
        padding="max_length",
        max_length=512,
        truncation=True
    )

    all_aligned_labels = []
    for i, labels in enumerate(examples["ner_tags"]):
        aligned_labels = []
        previous_word_id = None

        for word_id in tokenized_inputs.word_ids(batch_index=i):
            if word_id is None:
                aligned_labels.append(-100)
            elif word_id != previous_word_id:
                aligned_labels.append(labels[word_id])
            else:
                aligned_labels.append(-100)
            previous_word_id = word_id

        all_aligned_labels.append(aligned_labels)

    tokenized_inputs["labels"] = all_aligned_labels
    return tokenized_inputs

# Apply to all splits
tokenized_dataset = dataset.map(
    tokenize_and_align_dataset,
    batched=True,
    remove_columns=dataset["train"].column_names
)

print(tokenized_dataset)
print("\nFeatures:", tokenized_dataset["train"].features)
print("Sample label length:", len(tokenized_dataset["train"][0]["labels"]))

Map:   0%|          | 0/5433 [00:00<?, ? examples/s]

Map:   0%|          | 0/924 [00:00<?, ? examples/s]

Map:   0%|          | 0/941 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 5433
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 924
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 941
    })
})

Features: {'input_ids': List(Value('int32')), 'token_type_ids': List(Value('int8')), 'attention_mask': List(Value('int8')), 'labels': List(Value('int64'))}
Sample label length: 512


input_ids       → the tokenized text as numbers BERT understands
token_type_ids  → all zeros for single sentence tasks (used for sentence pairs)
attention_mask  → 1 for real tokens, 0 for padding tokens (tells BERT what to ignore)
labels          → your aligned BIO labels, -100 for ignored tokens

In [9]:
tokenized_dataset.save_to_disk("../data/tokenized_ncbi_disease")
print("Dataset saved successfully!")

Saving the dataset (0/1 shards):   0%|          | 0/5433 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/924 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/941 [00:00<?, ? examples/s]

Dataset saved successfully!
